# AudioMNIST ±1 LSB matching %5/%10 — ham dalga formu CNN (T4 GPU)

**Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**, sonra **Tümünü çalıştır**.

- AudioMNIST (MIT) sabit commit'ten Colab içinde indirilir; senin bir şey yüklemen gerekmez.
- Yalnızca daha önce açılmış 30 konuşmacı çıkarılır ve her dosyanın SHA-256'sı kontrol edilir. Saklı 30 konuşmacının baytları hiç okunmaz.
- Roller CPU çalışmasıyla aynıdır: 15 eğitim, 5 kalibrasyon, 10 test konuşmacısı (her biri 500 klip). Kalibrasyon steganları önceki çalışmalarla birebir aynıdır (digest kontrolü).
- Eğitim müfredatı %40 → %20 → %10 → %5. %40 aşaması pozitif kontroldür: kalibrasyonda %80'i geçemezse eğitim durur.
- Her epoch sonunda Drive'a (`MyDrive/steg-audiomnist-cnn`) checkpoint yazılır. Bağlantı koparsa **Tümünü çalıştır** ile kaldığı yerden devam eder (veri yeniden indirilir, eğitim devam eder).
- Test yalnızca bir kez ve yalnızca kalibrasyonda %80'i geçen seçimler için çalışır. Toplam süre tahmini: 1–1,5 saat.

**Bitince:** son hücrenin indirdiği `audiomnist-cnn-results.zip` dosyasını bana gönder (yalnızca JSON özetleri içerir, ses veya model yok).

In [ ]:
import os
import subprocess

from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/steg-audiomnist-cnn"
os.makedirs(OUT, exist_ok=True)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
import torch

assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı türünü T4 GPU yap"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
%%bash
set -euo pipefail
SHA=e441704997e9f121e0faf7e66a4dc05c4cdd6e6b
if [ ! -d /content/steganography/.git ]; then
  git init -q /content/steganography
  git -C /content/steganography remote add origin https://github.com/erkanrzgc/steganography
fi
cd /content/steganography
git fetch -q --depth 1 origin "$SHA"
git checkout -q FETCH_HEAD
test "$(git rev-parse HEAD)" = "$SHA"
git log -1 --format='%h %s'

In [ ]:
%%bash
set -euo pipefail
cd /content
URL=$(python -c "import json; print(json.load(open('/content/steganography/notebooks/audiomnist_wav_cnn_layout_20261118.json'))['url'])")
if [ ! -f audiomnist.zip ]; then
  curl -sSL --fail --retry 3 -o audiomnist.part "$URL"
  mv audiomnist.part audiomnist.zip
fi
ls -l audiomnist.zip

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
OUT=/content/drive/MyDrive/steg-audiomnist-cnn
cd /content/steganography
python -m research.wav_cnn_audiomnist prepare --layout notebooks/audiomnist_wav_cnn_layout_20261118.json \
  --data /content/amnist --archive /content/audiomnist.zip

## Eğitim

Her satır bir epoch: payload oranı, kayıp, kalibrasyon BA ve AUC. %40 ve %20 aşamaları hızlı yükselmeli; %10 ve %5 asıl hedef.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
OUT=/content/drive/MyDrive/steg-audiomnist-cnn
cd /content/steganography
python -m research.wav_cnn_audiomnist train --layout notebooks/audiomnist_wav_cnn_layout_20261118.json \
  --data /content/amnist --out "$OUT" --device cuda

## Tek seferlik test

Bu hücre yalnızca `choice-freeze.json` varsa çalışır. Kalibrasyonda %80'in altında kalan oranlar test edilmez. Test sonucu bir kez yazılır; tekrar çalıştırmak hata verir, bu beklenen davranıştır.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
OUT=/content/drive/MyDrive/steg-audiomnist-cnn
cd /content/steganography
if [ ! -f "$OUT/choice-freeze.json" ]; then
  echo "Seçim dondurulmadı (pozitif kontrol başarısız ya da eğitim bitmedi); test atlandı."
elif [ -f "$OUT/test-result.json" ]; then
  echo "Test zaten bir kez çalıştı; sonuç korunuyor."
else
  python -m research.wav_cnn_audiomnist test --layout notebooks/audiomnist_wav_cnn_layout_20261118.json \
    --data /content/amnist --out "$OUT" --device cuda
fi

In [ ]:
import json
import os
import zipfile

from google.colab import files

names = ["history.json", "choice-freeze.json", "test-result.json", "control-failed.json"]
archive = "/content/audiomnist-cnn-results.zip"
with zipfile.ZipFile(archive, "w") as bundle:
    for name in names:
        path = os.path.join(OUT, name)
        if os.path.exists(path):
            bundle.write(path, name)
            if name != "history.json":
                print(name, json.dumps(json.load(open(path)), indent=1)[:2000])
files.download(archive)